# Vapor-Eyes 04 — Attribution: candidate well pads (TX RRC)

**Step 4: tie the quantified plume to infrastructure.** NB03 located each EMIT plume's
max-concentration origin; here we shortlist the **candidate super-emitter wells** near
that origin from the Texas Railroad Commission and name their operators:

- **Stage** TX RRC well surface-hole locations (`WellSHL`) for the AOI with `WellsDownloader` (open ArcGIS REST → one merged GeoJSON on the Volume).
- **Read** the wells (`geojson_gbx`) — each carries its API number, operator (`CompanyName`), lease and field.
- **Rank** the nearest **candidate wells** to each plume origin with Databricks-native `st_distancesphere` over `st_point` / `st_geomfromwkb`.

Attribution is not a single closest pin: the plume's origin plus the **prevailing wind**
point to the source, so we shortlist the nearest wells and note the true super-emitter is
the one **upwind** of the origin (EMIT reports the wind it used; the plume's shape shows the
transport direction). Definitive attribution needs wind-transport modeling.

**Result:** each quantified plume tied to a short-list of candidate operator well pads.

---
_Last Modified:_ July 11, 2026

![EMIT plume origin → TX RRC WellSHL for the AOI → nearest candidate wells by st_distancesphere → operator shortlist](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/diagrams/vapor-eyes/vapor-eyes-04.png)

In [ ]:
%run ./config_nb

In [ ]:
K_CANDIDATES = 5   # nearest wells to shortlist per plume

## 1. Plume origins (from NB03)

Each EMIT plume's max-concentration point (`lon_max`/`lat_max`) from NB03's
`emit_plumes`, with the wind EMIT used (`wind_speed_ms`; the max-concentration point
marks the likely source, the plume drifts downwind). Run NB03 first — this reads its table.

In [ ]:
assert spark.catalog.tableExists("emit_plumes"), (
    "emit_plumes not found — run notebook 03 (EMIT quantification) first."
)
plumes = spark.table("emit_plumes").select(
    "plume_id", "max_conc_ppmm", "emission_rate_kg_hr", "wind_speed_ms",
    "lon_max", "lat_max",
)
print(f"emit_plumes: {plumes.count():,} plume origin(s)")
plumes.orderBy(F.col("max_conc_ppmm").desc()).limit(5).display()

## 2. Stage TX RRC well surface-hole locations

`WellsDownloader` pages the open TX RRC `WellSHL` ArcGIS FeatureServer for wells
intersecting the AOI (native EPSG:2277 → WGS84), merges them into one GeoJSON on the
Volume, and `geojson_gbx` reads it — `well_geom` is each surface hole as a WKB point.

In [ ]:
if FORCE_REBUILD or not spark.catalog.tableExists("wells_shl"):
    wells_dl = wells.download(get_aoi_bbox(), WELLS_DIR, spark=spark)
    print(f"... staged {wells_dl.first()['feature_count']:,} wells -> {WELLS_DIR}")
    wells_raw = wells.read(WELLS_DIR).select(
        F.col("API").cast("string").alias("api"),
        F.col("CompanyName").alias("operator"),
        F.col("LeaseName").alias("lease"),
        F.col("WellNbr").alias("well_no"),
        F.col("FieldName").alias("field"),
        F.col("County").alias("county"),
        F.col("WellURL").alias("well_url"),
        F.col("geom_0").alias("well_geom"),
    )
    finalize_delta(wells_raw, "wells_shl", do_display=False)
else:
    print("... wells_shl exists; skipping download (FORCE_REBUILD=False)")
_w = spark.table("wells_shl")
print(f"wells_shl: {_w.count():,} wells")
_w.drop("well_geom").limit(5).display()  # drop WKB geom + limit for GitHub ipynb

## 3. Nearest candidate wells per plume (Databricks-native ST)

Build the plume origin as `st_point(lon_max, lat_max)` and each well as
`st_geomfromwkb(well_geom)`; `st_distancesphere` gives the great-circle distance in
metres, and a per-plume `row_number()` window keeps the **K nearest** as candidate
sources — landing `plume_candidate_wells` (rank 1 = closest). The leading candidate is
the nearest well upwind of the origin; wind transport decides among the shortlist.

In [ ]:
from pyspark.sql.window import Window

p_pt = plumes.withColumn("plume_pt", F.expr("st_point(lon_max, lat_max)"))
w_pt = spark.table("wells_shl").select(
    "api", "operator", "lease", "well_no", "field", "county", "well_url",
    F.expr("st_geomfromwkb(well_geom)").alias("well_pt"),
    F.expr("st_x(st_geomfromwkb(well_geom))").alias("well_lon"),
    F.expr("st_y(st_geomfromwkb(well_geom))").alias("well_lat"),
)
paired = p_pt.crossJoin(w_pt).withColumn(
    "dist_m", F.expr("st_distancesphere(plume_pt, well_pt)")
)
candidates = (
    paired.withColumn(
        "rank", F.row_number().over(Window.partitionBy("plume_id").orderBy("dist_m"))
    )
    .filter(F.col("rank") <= K_CANDIDATES)
    .drop("plume_pt", "well_pt")
)
finalize_delta(candidates, "plume_candidate_wells", do_display=False)
_c = spark.table("plume_candidate_wells")
print(f"plume_candidate_wells: {_c.count():,} rows ({K_CANDIDATES} per plume)")
_c.select(
    "plume_id", "rank", "operator", "lease", "field",
    F.round("dist_m", 0).alias("dist_m"),
).orderBy("plume_id", "rank").limit(5).display()

## 4. Plume origin and its candidate well pads

The best-attributed plume (closest candidate well), its `K` candidate wells (violet),
the surrounding AOI wells (grey), and the origin→candidate attribution links over a
CartoDB basemap. The candidates are the shortlist; wind transport pinpoints the emitter
among them. Rendered through the `INTERACTIVE_PLOTS` toggle: a static map by default
(labeled legend + leading-candidate callout), or a pannable MapLibre map when `True`.

In [ ]:
import geopandas as gpd

# Feature the plume whose nearest candidate is closest (best-attributed).
featured_id = (
    spark.table("plume_candidate_wells")
    .filter("rank = 1").orderBy("dist_m").first()["plume_id"]
)
cand = (
    spark.table("plume_candidate_wells")
    .filter(F.col("plume_id") == featured_id).orderBy("rank")
    .select("rank", "operator", "lease", "dist_m", "well_lon", "well_lat")
    .toPandas()
)
origin = spark.table("emit_plumes").filter(
    F.col("plume_id") == featured_id
).select("lon_max", "lat_max").first()
_lead = cand.iloc[0]
print(
    f"... plume {featured_id} — leading candidate {_lead['operator']} "
    f"({_lead['dist_m']:,.0f} m)"
)

# context wells in view: framing box spans the origin + its candidates
_lons = list(cand.well_lon) + [origin["lon_max"]]
_lats = list(cand.well_lat) + [origin["lat_max"]]
dpad = 0.02
ctx_pdf = (
    spark.table("wells_shl")
    .withColumn("lon", F.expr("st_x(st_geomfromwkb(well_geom))"))
    .withColumn("lat", F.expr("st_y(st_geomfromwkb(well_geom))"))
    .filter(
        f"lon BETWEEN {min(_lons) - dpad} AND {max(_lons) + dpad} AND "
        f"lat BETWEEN {min(_lats) - dpad} AND {max(_lats) + dpad}"
    )
    .select("lon", "lat").toPandas()
)
ctx_gdf = gpd.GeoDataFrame(
    ctx_pdf, geometry=gpd.points_from_xy(ctx_pdf.lon, ctx_pdf.lat), crs=4326
)
cand_gdf = gpd.GeoDataFrame(
    cand, geometry=gpd.points_from_xy(cand.well_lon, cand.well_lat), crs=4326
)
orig_gdf = gpd.GeoDataFrame(
    {"plume_id": [featured_id]},
    geometry=gpd.points_from_xy([origin["lon_max"]], [origin["lat_max"]]),
    crs=4326,
)

if INTERACTIVE_PLOTS:
    # MapLibre: pan/zoom the well field. Attribution links as a line layer; the
    # wells / candidates / origin as colored circle layers (drawn bottom -> top).
    from shapely.geometry import LineString

    from databricks.labs.gbx.vizx import plot_interactive, vector_layer

    links_gdf = gpd.GeoDataFrame(
        geometry=[
            LineString([(origin["lon_max"], origin["lat_max"]), (r.well_lon, r.well_lat)])
            for r in cand.itertuples()
        ],
        crs=4326,
    )
    layers = [vector_layer(links_gdf, color="#6B4FA0", label="attribution links")]
    if not ctx_gdf.empty:
        layers.append(vector_layer(ctx_gdf, color="#9AA6B2", label="TX RRC wells"))
    layers += [
        vector_layer(cand_gdf, color="#6B4FA0", label=f"{K_CANDIDATES} candidate wells"),
        vector_layer(orig_gdf, color="#C81E1E", label="plume origin"),
    ]
    plot_interactive(layers, center=[origin["lon_max"], origin["lat_max"]], zoom=12)
else:
    import contextily as cx
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 9))
    if not ctx_gdf.empty:
        ctx_gdf.to_crs(3857).plot(
            ax=ax, color="#9AA6B2", markersize=10, alpha=0.6, label="TX RRC wells"
        )
    _org = orig_gdf.to_crs(3857)
    _cwl = cand_gdf.to_crs(3857)
    for gx, gy in zip(_cwl.geometry.x, _cwl.geometry.y):  # origin -> candidate links
        ax.plot([_org.geometry.x.iloc[0], gx], [_org.geometry.y.iloc[0], gy],
                color="#6B4FA0", ls="--", lw=1.0, alpha=0.7, zorder=3)
    _cwl.plot(ax=ax, color="#6B4FA0", marker="o", markersize=120, zorder=5,
              edgecolor="white", label=f"{K_CANDIDATES} candidate wells")
    _org.plot(ax=ax, color="#C81E1E", marker="*", markersize=460, zorder=6,
              label="plume origin")
    _bx = list(_cwl.geometry.x) + [_org.geometry.x.iloc[0]]
    _by = list(_cwl.geometry.y) + [_org.geometry.y.iloc[0]]
    _m = max(max(_bx) - min(_bx), max(_by) - min(_by)) * 0.2 + 1500
    ax.set_xlim(min(_bx) - _m, max(_bx) + _m)
    ax.set_ylim(min(_by) - _m, max(_by) + _m)
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
    ax.set_axis_off()
    ax.legend(loc="upper right")
    ax.set_title(
        f"Plume {featured_id} — leading candidate {_lead['operator']} "
        f"({_lead['dist_m']:,.0f} m)"
    )

## What we built

- **`wells_shl`** (Delta) — TX RRC well surface-hole locations for the AOI (operator, lease, field, WKB point).
- **`plume_candidate_wells`** (Delta) — the K nearest candidate wells per plume, ranked by distance, with operator/lease/field.
- A **plume-to-candidate-wells map** over a basemap.

GeoBrix: `WellsDownloader`, `geojson_gbx`.
Databricks-native: `st_point`, `st_geomfromwkb`, `st_x` / `st_y`, `st_distancesphere`.

Next: **notebook 05** synthesizes the cascade into a shareable PMTiles portfolio.